# TRELLIS – Text/Image to 3D (بدون Flash Attention)
تشغيل [TRELLIS](https://github.com/JeffreyXiang/TRELLIS) على Google Colab.

✅ **يعمل على أي GPU (T4, A100...)** باستخدام `sdpa` (انتباه PyTorch الافتراضي).
✅ **تثبيت صحيح** للحزمة من المصدر الأصلي، لا مشاكل في المسارات.

In [ ]:
import torch, subprocess
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("الرجاء تفعيل GPU من Runtime → Change runtime type → GPU")
!nvidia-smi
gpu_name = subprocess.getoutput("nvidia-smi --query-gpu=name --format=csv,noheader")
print(f"GPU: {gpu_name}")
print("✅ سيتم استخدام SDPA (الانتباه الافتراضي).")

In [ ]:
import pkg_resources, sys

# دالة فحص سريعة
def need(pkg_name):
    try:
        pkg_resources.get_distribution(pkg_name)
        return False
    except pkg_resources.DistributionNotFound:
        return True

# الحزم الضرورية (بدون trellis نفسه)
reqs = ["spconv", "trimesh", "pygltflib", "viser", "omegaconf", "opencv-python", "imageio", "gradio", "huggingface_hub"]
for r in reqs:
    if need(r):
        !pip install -q {r}
        print(f"✔️ {r}")
    else:
        print(f"✅ {r} (موجود)")

# تثبيت مكتبة trellis من المستودع الأصلي (يدعم sdpa بدون flash-attn)
if need("trellis"):
    !pip install -q git+https://github.com/JeffreyXiang/TRELLIS.git
    print("✔️ تم تثبيت trellis من المصدر")
else:
    print("✅ trellis مثبتة مسبقاً")

print("\n✅ جميع الحزم جاهزة.")

In [ ]:
import torch
from trellis.pipelines import TrellisImageTo3DPipeline

# تحميل النموذج مع تعطيل flash-attn
pipeline = TrellisImageTo3DPipeline.from_pretrained(
    "JeffreyXiang/TRELLIS-image-large",
    torch_dtype=torch.float16,
    attn_implementation="sdpa"
)
pipeline.to("cuda")
print("✅ تم تحميل خط الأنابيب.")

In [ ]:
from PIL import Image
import requests

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/tiger.png"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
display(image)

In [ ]:
outputs = pipeline.run(image, seed=42)
gaussian = outputs.get('gaussian')
mesh = outputs.get('mesh')
print("🔹 تم التوليد.")

In [ ]:
from pathlib import Path
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

if mesh is not None:
    mesh.export(out_dir / "model.glb")
    print("✅ model.glb")
if gaussian is not None and hasattr(gaussian[0], 'save_ply'):
    gaussian[0].save_ply(out_dir / "gaussian.ply")
    print("✅ gaussian.ply")

In [ ]:
import viser
from google.colab.output import eval_js
from IPython.display import IFrame, display

server = viser.ViserServer(host="0.0.0.0", port=8080)

if gaussian is not None:
    try:
        from trellis.utils import render_utils
        render_utils.render_gaussian(server, gaussian[0])
    except ImportError:
        points = gaussian[0].xyz.detach().cpu().numpy().reshape(-1, 3)
        server.scene.add_point_cloud("/points", points=points, colors=(255,200,200), point_size=0.01)

url = eval_js(f"google.colab.kernel.proxyPort(8080)")
public_url = f"https://{url}"
print(f"🔗 رابط المشاهدة: {public_url}")
display(IFrame(src=public_url, width="100%", height="600px"))

## لماذا اختفى الخطأ `ModuleNotFoundError`؟
- **تثبيت `trellis` مباشرة** من مستودع JeffreyXiang/TRELLIS الرسمي (عبر `pip install git+...`).
- **لا حاجة** لاستنساخ مستودع PRITHIVSAKTHIUR أو التعامل مع مجلدات غامضة.
- **الانتباه الافتراضي (`sdpa`)** يعمل بكفاءة على أي GPU.

شغّل الخلايا بالترتيب.